# Sensitivity analysis - dependence on sample density, weights, and the Power Model exponent

Notebook that interactively reproduces `PIPELINE.md` / `FINAL_SPRINT_PLAN.md` P1-1. It tests how much the "Top Priority" list (`priority_class`, top 3% per country) and the fatality-reduction estimate (`delta_fatal_percent`, P0-1) depend on **assumptions we chose** (the sample-density threshold, the score-blending weights, the Power Model exponent).

All three checks simply call the functions in `src/sensitivity_analysis.py` directly, with no dedicated simplified logic here (always kept consistent with `safety_score.add_safety_score` / `fatality_reduction.add_fatal_reduction` themselves). It only reads `data/processed/segments_v_safe.parquet` (all 15,121 segments, committed), so no raw data or WorldPop rasters are needed.

In [1]:
import os
import sys
import warnings

if os.path.basename(os.getcwd) == "notebooks":
 os.chdir("..")
sys.path.insert(0, "src")
warnings.filterwarnings("ignore", category=UserWarning)

import geopandas as gpd
import pandas as pd

pd.set_option("display.width", 200)

gdf = gpd.read_parquet("data/processed/segments_v_safe.parquet")
valid = gdf[gdf["data_quality_flag"].isna]
print(f"loaded {len(gdf)} segments, {len(valid)} valid (scoreable)")

loaded 15121 segments, 14711 valid (scoreable)


## : Dependence on sample density

Recompute `priority_class` using only the segments remaining after excluding the bottom 25% of `sample_size_total` -- the "low confidence" candidate threshold from initial data exploration -- and look at the overlap (recall/Jaccard) with the original Top Priority list. If low-sample segments are what's driving the Top Priority list's conclusion, this should shuffle it substantially.

In [2]:
from sensitivity_analysis import sample_size_robustness

s = sample_size_robustness(valid)
print(f"sample_size_total threshold (25th pct): {s['sample_size_threshold']:.0f}")
print(f"segments excluded as low-sample: {s['n_excluded_low_sample_segments']} / {len(valid)}")
print(f"of baseline Top Priority (n={s['n_baseline']}), excluded as low-sample: {s['n_baseline_top_excluded_as_low_sample']}")
print(f"Top Priority after recompute: n={s['n_other']}, recall={s['recall_of_baseline']:.1%}, jaccard={s['jaccard']:.1%}")

sample_size_total threshold (25th pct): 49730
segments excluded as low-sample: 3678 / 14711
baseline Top Priority (n=562), excluded as low-sample: 58
Top Priority after recompute: n=473, recall=84.2%, jaccard=84.2%


**Conclusion:** even after excluding low-sample segments, over 80% of the conclusion (recall/Jaccard 84.2%) holds, supporting that the Top Priority list is not an artifact of low-confidence data.

## : Dependence on the score-blending weights

The weights in `safety_score.py` (misalignment 0.50 / exposure 0.35 / confidence 0.15) are an assumption set by this analysis team. Recompute the Top Priority list under 5 different weighting scenarios and compare the overlap against the baseline.

In [3]:
from sensitivity_analysis import weight_robustness

w = weight_robustness(valid)
w[["n_baseline", "n_other", "n_overlap", "recall_of_baseline", "jaccard"]]

,n_baseline,n_other,n_overlap,recall_of_baseline,jaccard
scenario,,,,,
baseline (0.50/0.35/0.15),562,562,562,1.000000,1.000000
equal weights (0.33/0.33/0.34),562,781,519,0.923488,0.629854
misalignment-heavy (0.70/0.20/0.10),562,562,562,1.000000,1.000000
exposure-heavy (0.30/0.55/0.15),562,569,562,1.000000,0.987698
confidence-heavy (0.40/0.30/0.30),562,781,519,0.923488,0.629854


**Conclusion:** as long as misalignment (the primary axis) carries a weight around 0.5, the Top Priority list is robust (Jaccard 98.8-100%). Bringing the three axes closer to equal (equal weights), or giving the data-confidence axis twice the baseline weight (confidence-heavy), drops Jaccard to as low as 63% -- the specific value chosen for the confidence weight (0.15) is defensible within the range tested by this sensitivity analysis, but a residual degree of arbitrariness cannot be fully ruled out.

## : Dependence on the Power Model exponent's confidence interval (linked to P0-1)

The fatality-reduction estimate (`delta_fatal_percent`, `src/fatality_reduction.py`) uses the point estimate of the Cameron & Elvik (2010) exponent (rural 4.1 / urban 2.6). No new formula is derived here; this simply aggregates the confidence-interval columns that `add_fatal_reduction` already computes per segment (`delta_fatal_percent_ci_low/high`, the exponent's 95% CI carried straight through the formula), restricted to priority segments in the "Review Needed" track with a positive reduction effect.

In [4]:
from sensitivity_analysis import power_model_sensitivity

p = power_model_sensitivity(valid)
p.round(1)

,n,ci_low_mean,point_estimate_mean,ci_high_mean
power_environment_used,,,,
rural_freeway,1835,83.9,90.5,93.7
urban_residential,1232,16.6,75.5,90.5


**Conclusion:** the rural (rural_freeway) confidence interval is narrow (83.9-93.7%), so the point estimate of 90.5% is a stable estimate. The urban (urban_residential) confidence interval, on the other hand, is very wide at 16.6-90.5%, so looking only at the 75.5% point estimate risks overconfidence -- this numerically confirms limitation (b) of the fatality-reduction section ("the urban benefit estimate is approximate; don't overtrust it"), and is the basis for the policy of presenting urban priority segments alongside their confidence interval rather than the point estimate alone.

## Summary

| Check | Target | Conclusion |
|---|---|---|
| Sample density | Top Priority list | 84.2% retained after excluding low-sample segments; not an artifact |
| Score-blending weights | Top Priority list | Robust as long as misalignment (primary axis) stays around 0.5; giving the confidence axis 2x+ the baseline weight swaps out nearly 40% |
| Power Model exponent | Fatality-reduction estimate | Rural CI is narrow and stable; urban CI is very wide, so the point estimate alone would overstate confidence |

See the "Sensitivity Analysis" and "Fatality Reduction Estimate" sections of `README.md` for details and sources.